In [ ]:
%pip install pandas

In [2]:
import pandas as pd
import numpy as np


df = pd.read_csv('data_base_copro.csv')
df.columns = df.columns.str.strip().str.lower()

cols_numeriques = ['lots_habitation', 'lots_parking', 'total_lots']
for col in cols_numeriques:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)



df_propre = df[df['lots_habitation'] >= 5].copy()
df_propre['ratio_habitation'] = df_propre['lots_habitation'] / df_propre['total_lots'].replace(0, np.nan)
df_propre = df_propre[df_propre['ratio_habitation'] >= 0.3].copy()



df_unique = df_propre.groupby(['adresse', 'ville', 'code_postal', 'dept_code']).agg({
    'lots_habitation': 'sum',
    'lots_parking': 'sum',
    'nom_syndic': 'first',   
    'type_syndic': 'first'   
}).reset_index()


df_unique['score_immeuble'] = df_unique['lots_parking'] / df_unique['lots_habitation'].replace(0, 1)

df_cibles = df_unique[(df_unique['score_immeuble'] > 1) & (df_unique['score_immeuble'] <= 5)].copy()

df_cibles = df_cibles.dropna(subset=['nom_syndic'])

mots_a_virer = ['non connu', 'non renseigné', 'nc', '0', 'identité non partagée en open data']
df_cibles = df_cibles[~df_cibles['nom_syndic'].str.lower().isin(mots_a_virer)]


kpi4_syndics = df_cibles.groupby('nom_syndic').agg(
    nb_immeubles_cibles=('adresse', 'count'),       
    total_places_parking=('lots_parking', 'sum')    
).reset_index()

kpi4_syndics = kpi4_syndics.sort_values('nb_immeubles_cibles', ascending=False).reset_index(drop=True)

print(" KPI 4 - TOP des syndics à démarcher :")
display(kpi4_syndics.head(10))

 KPI 4 - TOP des syndics à démarcher :


,nom_syndic,nb_immeubles_cibles,total_places_parking
0,LAMY,2309,130796
1,SOCIETE D ETUDES ET DE REALISATION DE GESTION ...,359,26362
2,FONCIA ALSACE,302,10444
3,PICHET IMMOBILIER SERVICES,277,20258
4,FONCIA LACOMBE VAUCELLES,261,18409
5,FONCIA TOULON,241,14253
6,CABINET LOISELET PERE FILS ET DAIGREMONT,235,22797
7,FONCIA NORMANDIE,226,13671
8,FONCIA MARNE LA VALLEE,223,15480
9,CABINET THIERRY,206,12602
